# 01 — Build housing-pressure indices

This notebook constructs the measurement layer of the project. The aim is to answer two descriptive questions before fitting any explanatory model:

1. How far have local rents moved away from local incomes?
2. How much of the housing stock is exposed to short-term accommodation, and how intense is tourist demand?

The notebook keeps **registered AL**, **platform-listed entire homes**, and **active-listing proxies** separate. It does not treat registration as proof of active tourist use.

The core measures are

\[
THCR_{it}=1000\frac{AL_{it}}{H_{it}},
\qquad
LHDI_{it}=100\frac{R_{it}/Y_{it}}{R_{i0}/Y_{i0}},
\qquad
TI_{it}=\frac{N_{it}}{P_{it}}.
\]

The natural common start is 2017 because the INE local rent series begins there.

**Interpretation note:** LHDI is a normalised rent-price-to-income proxy. The rent numerator is EUR/m² for new leases and the income denominator is per taxpayer, so it must not be presented as the literal share of a household's income spent on rent.

## Scientific rules

- Preserve raw responses before cleaning.
- Do not turn suppressed or missing observations into zero.
- Keep one common geography definition for the core panel.
- Inspect every INE non-geography dimension before selecting a `Total` category.
- Historical RNAL exposure requires dated snapshots or valid registration/cancellation history. A present registry alone is not a historical stock series.
- The output of this notebook is measurement, not causal evidence.

In [ ]:
from __future__ import annotations

import sys
import tomllib
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from housing_tourism.data import (  # noqa: E402
    INEClient,
    canonicalise_ine_measure,
    describe_dimensions,
    infer_total_filters,
    save_flat_ine_indicator,
)
from housing_tourism.panel import add_core_indices, merge_canonical_series  # noqa: E402
from housing_tourism.rnal import (  # noqa: E402
    RNALClient,
    surviving_registration_panel,
)

RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
FIGURES = ROOT / "outputs" / "figures"
TABLES = ROOT / "outputs" / "tables"
for path in (RAW, PROCESSED, FIGURES, TABLES):
    path.mkdir(parents=True, exist_ok=True)

with (ROOT / "config" / "sources.toml").open("rb") as handle:
    SOURCES = tomllib.load(handle)

pd.set_option("display.max_columns", 80)

## 1. Source catalogue

In [ ]:
source_table = pd.DataFrame(
    [
        {
            "measure": key,
            "indicator": value.get("indicator"),
            "description": value["name"],
            "url": value["url"],
        }
        for key, value in SOURCES["ine"].items()
    ]
)
source_table

The first version deliberately uses the NUTS-2013 variants of the INE series, because rent, income, housing stock, population and tourism all have compatible NUTS-2013 indicators. Geography changes should be handled explicitly in a later extension rather than mixed silently into version 0.1.

## 2. Download and cache the INE indicators

In [ ]:
DOWNLOAD_INE = False  # Set True when running with network access.
REFRESH_CACHE = False

indicator_files: dict[str, Path] = {}
client = INEClient(RAW / "ine")

for measure, source in SOURCES["ine"].items():
    output_path = RAW / "ine" / f"{source['indicator']}_flat.parquet"
    indicator_files[measure] = output_path
    if DOWNLOAD_INE:
        frame = save_flat_ine_indicator(
            client,
            source["indicator"],
            output_path,
            refresh=REFRESH_CACHE,
        )
        print(measure, frame.shape, output_path)

indicator_files

If `DOWNLOAD_INE=False`, this notebook expects the cached Parquet files produced by a previous run. The exact raw JSON responses remain in `data/raw/ine/` for auditability.

In [ ]:
missing_cache = [name for name, path in indicator_files.items() if not path.exists()]
if missing_cache:
    print(
        "Missing cached INE indicators:",
        missing_cache,
        "\nSet DOWNLOAD_INE=True when network access is available."
    )

## 3. Inspect and canonicalise the official series

Dimension labels must be inspected before selecting totals. The helper accepts a `Total` category only when it is unambiguous. Geography level must also be verified against INE metadata rather than inferred from code length.

In [ ]:
ine_frames: dict[str, pd.DataFrame] = {}
for measure, path in indicator_files.items():
    if path.exists():
        ine_frames[measure] = pd.read_parquet(path)
        print(f"\n### {measure}")
        print(describe_dimensions(ine_frames[measure], max_values=30))

dimension_filters = {measure: infer_total_filters(frame) for measure, frame in ine_frames.items()}
VALUE_NAMES = {
    "rent": "rent_eur_m2",
    "income": "income_eur",
    "housing_stock": "housing_stock",
    "population": "resident_population",
    "overnight_stays": "overnight_stays",
}
canonical = {
    measure: canonicalise_ine_measure(
        frame, value_name=VALUE_NAMES[measure], filters=dimension_filters[measure], minimum_year=2017
    )
    for measure, frame in ine_frames.items()
}

## 4. RNAL: official current snapshot and historical-survival proxy

The official Turismo de Portugal ArcGIS layer publishes `DataRegisto`, municipality and parish fields, but no cancellation/end date. The notebook therefore keeps the observed current snapshot separate from a deliberately named surviving-registration proxy. The proxy is not historical active stock.

In [ ]:
RNAL_SNAPSHOT_DATE = "2026-08-29"
DOWNLOAD_RNAL = False
REFRESH_RNAL = False

rnal_client = RNALClient(RAW / "rnal")
rnal_snapshot_path = RAW / "rnal" / RNAL_SNAPSHOT_DATE / "rnal.parquet"
if DOWNLOAD_RNAL:
    rnal_snapshot = rnal_client.fetch_current_snapshot(snapshot_date=RNAL_SNAPSHOT_DATE, refresh=REFRESH_RNAL)
elif rnal_snapshot_path.exists():
    rnal_snapshot = pd.read_parquet(rnal_snapshot_path)
else:
    rnal_snapshot = None

if rnal_snapshot is not None:
    rnal_survivor_proxy = surviving_registration_panel(
        rnal_snapshot, start_year=2017, end_year=int(RNAL_SNAPSHOT_DATE[:4])
    )
    rnal_survivor_proxy.to_parquet(PROCESSED / "rnal_surviving_registrations_annual.parquet", index=False)

## 5. Build the municipality-year panel

In [ ]:
AL_PANEL_PATH = PROCESSED / "al_units_annual.parquet"
panel: pd.DataFrame | None = None
if canonical and AL_PANEL_PATH.exists():
    al_annual = pd.read_parquet(AL_PANEL_PATH)
    panel = merge_canonical_series([
        (canonical["rent"], "rent_eur_m2"),
        (canonical["income"], "income_eur"),
        (canonical["housing_stock"], "housing_stock"),
        (canonical["population"], "resident_population"),
        (canonical["overnight_stays"], "overnight_stays"),
        (al_annual, "al_units"),
    ])
    panel = add_core_indices(panel, base_year=2017, al_col="al_units")
    panel["al_exposure_definition"] = "dated_active_stock"
    panel.to_parquet(PROCESSED / "municipality_housing_panel.parquet", index=False)

## Handover to Notebook 02

Notebook 02 should only be run after municipality geography, common years, missingness and the historical AL exposure construction have been reviewed. A visually strong relationship between THCR and LHDI is not a causal result.